# PE-U4 — Ejecucion del experimento en Google Colab

Las celdas se ejecutan secuencialmente.
Requiere cargar el paquete de código en la celda 2..

## Celda 1 — Verificación del entorno de ejecución

Colab asigna maquinas distintas. Necesitamos saber cuantos nucleos tiene,
porque de eso depende el experimento de Amdahl.

In [ ]:
import multiprocessing
nucleos = multiprocessing.cpu_count()
print("Nucleos disponibles:", nucleos)
print("Version de Java:")
!java -version
print()
if nucleos < 4:
    print("=" * 70)
    print("AVISO: esta maquina solo tiene " + str(nucleos) + " nucleos.")
    print("El barrido de 1/2/4 executors no mostrara ganancia real en N=4,")
    print("porque no hay 4 nucleos fisicos donde repartir el trabajo.")
    print()
    print("Opciones:")
    print("  a) Continuar igual y DOCUMENTARLO como limitacion experimental")
    print("     (es una respuesta valida y honesta para el informe).")
    print("  b) Entorno de ejecucion > Cambiar tipo de entorno > probar otra opcion,")
    print("     o usar Databricks Community Edition, que da mas nucleos.")
    print("=" * 70)
else:
    print("Perfecto: hay nucleos suficientes para el barrido de 1/2/4 executors.")

## Celda 2 — Carga del proyecto

Al ejecutar esta celda aparece un boton **"Elegir archivos"**.
Se carga el paquete de código del experimento.

> **Nota tecnica.** `Compress-Archive` de Windows guarda las rutas internas con
> barra invertida (`core\\config.py`). En Linux eso no crea carpetas, sino archivos
> con ese nombre literal. Esta celda normaliza los separadores al descomprimir,
> asi que el ZIP de Windows funciona sin cambios.

In [ ]:
import os, shutil, zipfile
from google.colab import files

DESTINO = "/content/proyecto"
if os.path.exists(DESTINO):
    shutil.rmtree(DESTINO)
os.makedirs(DESTINO)

subidos = files.upload()
nombre_zip = list(subidos.keys())[0]

# Descomprimir normalizando los separadores de Windows (\\ -> /)
with zipfile.ZipFile(nombre_zip, "r") as z:
    for entrada in z.namelist():
        destino_rel = entrada.replace("\\", "/")
        if destino_rel.endswith("/"):
            continue
        destino_abs = os.path.join(DESTINO, destino_rel)
        os.makedirs(os.path.dirname(destino_abs), exist_ok=True)
        with z.open(entrada) as origen, open(destino_abs, "wb") as salida:
            shutil.copyfileobj(origen, salida)

# Buscar la carpeta que contiene core/ y main.py
RAIZ = None
for raiz, dirs, archivos in os.walk(DESTINO):
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".git", ".venv")]
    if "core" in dirs and "main.py" in archivos:
        RAIZ = raiz
        break

if RAIZ is None:
    print("NO se encontro el proyecto. Esto es lo que se descomprimio:")
    for raiz, dirs, archivos in os.walk(DESTINO):
        print(" ", raiz, "->", dirs, archivos[:8])
else:
    print("Proyecto encontrado en:", RAIZ)
    print()
    modulos = sorted(a for a in os.listdir(os.path.join(RAIZ, "core"))
                     if a.endswith(".py"))
    print("Modulos en core/ (" + str(len(modulos)) + "):")
    for m in modulos:
        print("   ", m)

## Celda 3 — Instalar PySpark

Tarda uno o dos minutos.

Es normal que aparezca un **ERROR rojo** sobre `dataproc-spark-connect`:
es un paquete preinstalado de Colab que no usamos, y no afecta al experimento.
Lo que importa es la ultima linea, que debe decir `PySpark: 3.5.0`.

In [ ]:
!pip install -q pyspark==3.5.0 faker
print()
import pyspark
print("PySpark:", pyspark.__version__)

## Celda 4 — Verificar que los modulos cargan

Usa la ruta detectada en la celda 2. Si esta celda pasa, el codigo esta bien.

In [ ]:
import sys

if "RAIZ" not in dir() or RAIZ is None:
    raise RuntimeError("Ejecuta primero la celda 2: no hay ruta del proyecto.")

if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
os.chdir(RAIZ)

from core import config, dataset, benchmark, amdahl, figuras, exportar_latex
from core import transformaciones_pandas, transformaciones_spark

print("Los 8 modulos cargaron correctamente.")
print("Directorio de trabajo:", os.getcwd())
print()
print("Configuracion del experimento:")
print("  Registros de reservas : {:,}".format(config.N_RESERVAS))
print("  Repeticiones          :", config.REPETICIONES)
print("  Executors a probar    :", config.CORES_TEST)
print("  Directorio de salida  :", config.BASE)

## Celda 5 — Ejecutar el experimento completo

La celda larga: entre **8 y 20 minutos**. Duración aproximada: 9 minutos.

Veras pasar los cuatro pasos de la guia. Al final imprime la tabla de Amdahl
y genera las figuras.

In [ ]:
import time

t_inicio = time.time()
!cd "{RAIZ}" && python main.py
print()
print("Tiempo total: {:.1f} minutos".format((time.time() - t_inicio) / 60))

## Celda 6 — Revisar los resultados

Muestra los numeros clave y las tres figuras.

In [ ]:
import json
from IPython.display import Image, display

ruta_json = os.path.join(RAIZ, "salida_pe_u4", "resultados.json")
with open(ruta_json) as f:
    res = json.load(f)

print("=" * 60)
print("ANALISIS DE AMDAHL")
print("=" * 60)
a = res["amdahl"]
print("Fraccion paralelizable p   : {:.4f}".format(a["p_paralelizable"]))
print("Fraccion serial (1-p)      : {:.4f}".format(a["fraccion_serial"]))
print("Speedup teorico maximo     : {:.3f}x".format(a["S_max"]))
print("Procesadores para el 90%   :", int(a["N_90_entero"]))
print()
print("=" * 60)
print("VERIFICACION pandas vs PySpark")
print("=" * 60)
for tx, v in res["verificacion"].items():
    estado = "IDENTICO" if v["identico"] else "REVISAR"
    print("  {}: {}  ({:,} filas)".format(tx, estado, v["filas_pandas"]))

for fig in ["fig_a_tiempos.png", "fig_b_speedup_amdahl.png", "fig_c_eficiencia.png"]:
    print()
    display(Image(os.path.join(RAIZ, "salida_pe_u4", "figs", fig)))

## Celda 7 — Exportación de resultados

Genera un ZIP con los CSV, las figuras a 300 DPI, las tablas LaTeX y el JSON.

In [ ]:
import shutil
from google.colab import files

ruta_zip = shutil.make_archive(
    "/content/PE_U4_resultados", "zip", os.path.join(RAIZ, "salida_pe_u4")
)
print("ZIP generado: {:.1f} MB".format(os.path.getsize(ruta_zip) / 1e6))
files.download(ruta_zip)

## Celda 8 — Captura para la carpeta `evidencia/`

Documenta la máquina, las versiones de las librerías y los parámetros del
experimento. La salida de esta celda se conserva en `evidencia/` como
constancia de que la ejecución se realizó en Google Colab, conforme al
entorno establecido en la guía de práctica.

In [ ]:
import platform, multiprocessing, pyspark, pandas as pd, numpy as np
from datetime import datetime

print("=" * 62)
print("  EVIDENCIA DE EJECUCION - PE-U4")
print("=" * 62)
print("  Fecha y hora    :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("  Entorno         : Google Colab")
print("  Sistema         :", platform.platform())
print("  Python          :", platform.python_version())
print("  Nucleos de CPU  :", multiprocessing.cpu_count())
print("  PySpark         :", pyspark.__version__)
print("  pandas          :", pd.__version__)
print("  NumPy           :", np.__version__)
print("-" * 62)
print("  Registros       : {:,}".format(res["configuracion"]["n_reservas"]))
print("  Semilla         :", res["configuracion"]["semilla"])
print("  Repeticiones    :", res["configuracion"]["repeticiones"])
print("=" * 62)
!free -h